In [1]:
import numpy as np
from numba import cuda

In [2]:
@cuda.jit
def bitonic_kernel(arr, j, k):
    i = cuda.threadIdx.x   # only one block, very small array
    ixj = i ^ j

    if ixj > i:
        if (i & k) == 0:
            # ascending
            if arr[i] > arr[ixj]:
                arr[i], arr[ixj] = arr[ixj], arr[i]
        else:
            # descending
            if arr[i] < arr[ixj]:
                arr[i], arr[ixj] = arr[ixj], arr[i]

In [3]:
def bitonic_sort_small():
    arr = np.array([3, 7, 4, 8, 6, 2, 1, 5], dtype=np.int32)

    print("Before:", arr)

    d_arr = cuda.to_device(arr)

    for k in [2, 4, 8]:
        for j in [k//2, k//4, k//8]:
            if j > 0:
                bitonic_kernel[1, 8](d_arr, j, k)
                cuda.synchronize()

    sorted_arr = d_arr.copy_to_host()
    print("After :", sorted_arr)

In [4]:
bitonic_sort_small()

Before: [3 7 4 8 6 2 1 5]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


After : [1 2 3 4 5 6 7 8]
